# Add missing IFRS S1/S2 disclosure data (synthetic, per-bank)

Creates **new CSV tables in `gen_data`** for the disclosure areas that genuinely had no source data, derived per-bank from each bank's existing records. Existing CSVs are **not modified**; the generator only adds files. Deterministic and safe to re-run.

**Every row is flagged `is_synthetic=True`** with a `data_source` marker. These are placeholder estimates to let the pipeline exercise these requirements end-to-end — for a real entity they must be replaced with the bank's actual figures and narrative. Treat them as test fixtures, not disclosures.

New tables and the requirement each fills:
- `scope3_categories.csv` - Scope 3 by GHG Protocol category, with non-relevant ones explicitly excluded (S2 29(a)(i)(3) & (vi)).
- `transition_plan.csv` - transition-plan assumptions, dependencies, resourcing, prior-period progress (S2 14).
- `climate_financial_effects.csv` - current & anticipated effects mapped to financial-statement line items (S2 15-16).
- `resilience_assessment.csv` - capacity to adjust, asset redeployment, financial flexibility, uncertainties (S2 22(a)).
- `ghg_methodology.csv` - measurement approach/inputs/assumptions per scope (S2 29(a)(iii)).
- `scope12_consolidation.csv` - Scope 1&2 consolidated-group vs other-investees split (S2 29(a)(iv)).

Set `GEN_DATA` if your folder isn't `gen_data`. After running, re-run `data_prep` so the payload picks up the new tables (the exporter will need wiring to read them — say the word and I'll add that).

In [2]:
# === ADD MISSING IFRS S1/S2 DISCLOSURE DATA (synthetic, per-bank) ============
# Creates NEW csv tables in gen_data for the disclosure areas that genuinely had no
# source data, derived per-bank from each bank's existing records and clearly flagged
# as synthetic estimates (is_synthetic=True, data_source set). Existing CSVs are NOT
# modified. Deterministic (fixed seed). Safe to re-run.
import json, hashlib
from pathlib import Path
import pandas as pd

GEN_DATA = "gen_data"
gd = Path(GEN_DATA)

def rd(name):
    f = gd / f"{name}.csv"
    return pd.read_csv(f) if f.exists() else None

fin   = rd("financial_summary")
emp   = rd("employees")
tgt   = rd("targets")
scen  = rd("climate_scenarios")
rr    = rd("climate_risk_register")
opp   = rd("climate_opportunities")
if fin is None:
    raise FileNotFoundError("financial_summary.csv not found — set GEN_DATA to your gen_data folder.")

banks = sorted(fin["bank_id"].unique())
years = sorted(fin["reporting_year"].unique())
YEAR  = max(years)

def jit(key, pct=0.05):           # deterministic +/- jitter from a stable hash
    h = int(hashlib.md5(key.encode()).hexdigest(), 16)
    return 1.0 + ((h % 1000) / 1000.0 * 2 - 1) * pct

def fin_val(bank, year, col):
    r = fin[(fin["bank_id"] == bank) & (fin["reporting_year"] == year)]
    return float(r[col].iloc[0]) if not r.empty and col in r and pd.notna(r[col].iloc[0]) else None

def headcount(bank, year):
    if emp is None: return None
    r = emp[(emp["bank_id"] == bank) & (emp["reporting_year"] == year)]
    return float(r["total_headcount"].iloc[0]) if not r.empty else None

# ---- 1) scope3_categories.csv  (IFRS S2 para 29(a)(i)(3) & (vi)) -----------
# Operational Scope 3 estimated by spend-/headcount-based factors; categories 6 and 15
# are reported elsewhere; non-relevant categories explicitly excluded with a reason.
S3_FACTORS = {  # tCO2e per EUR m revenue, except cat 7 (per employee)
    1: ("Purchased goods and services", 0.60, "spend_based_eeio"),
    2: ("Capital goods",                0.15, "spend_based_eeio"),
    3: ("Fuel- and energy-related activities", 0.12, "average_data_method"),
    5: ("Waste generated in operations", 0.02, "average_data_method"),
    7: ("Employee commuting",           0.55, "average_data_method"),  # per-employee
}
S3_ELSEWHERE = {6: "Business travel (reported in operational Scope 3 travel)",
                15: "Financed emissions (reported in the financed-emissions section)"}
S3_EXCLUDED = {4: "Upstream transportation and distribution", 8: "Upstream leased assets",
               9: "Downstream transportation and distribution", 10: "Processing of sold products",
               11: "Use of sold products", 12: "End-of-life treatment of sold products",
               13: "Downstream leased assets", 14: "Franchises"}
s3 = []
for b in banks:
    for y in years:
        rev, hc = fin_val(b, y, "total_revenue_meur"), headcount(b, y)
        for cat, (name, factor, method) in S3_FACTORS.items():
            base = (hc if cat == 7 else rev)
            em = round(base * factor * jit(f"{b}{y}{cat}"), 1) if base else None
            s3.append(dict(scope3_id=f"S3-{b}-{y}-C{cat:02d}", bank_id=b, reporting_year=y,
                           category_number=cat, category_name=name, included_flag=True,
                           emissions_tco2e=em, calculation_method=method,
                           data_source="spend_based_estimate", pcaf_data_quality_score=4,
                           exclusion_reason=None, is_synthetic=True))
        for cat, note in S3_ELSEWHERE.items():
            s3.append(dict(scope3_id=f"S3-{b}-{y}-C{cat:02d}", bank_id=b, reporting_year=y,
                           category_number=cat, category_name=note, included_flag=True,
                           emissions_tco2e=None, calculation_method="reported_separately",
                           data_source="cross_reference", pcaf_data_quality_score=None,
                           exclusion_reason=None, is_synthetic=True))
        for cat, name in S3_EXCLUDED.items():
            s3.append(dict(scope3_id=f"S3-{b}-{y}-C{cat:02d}", bank_id=b, reporting_year=y,
                           category_number=cat, category_name=name, included_flag=False,
                           emissions_tco2e=None, calculation_method=None,
                           data_source=None, pcaf_data_quality_score=None,
                           exclusion_reason="Not relevant to a banking business model / assessed not material",
                           is_synthetic=True))
scope3_categories = pd.DataFrame(s3).sort_values(["bank_id","reporting_year","category_number"])

# ---- 2) transition_plan.csv  (IFRS S2 para 14(a)(iv),(b),(c)) --------------
tp = []
for b in banks:
    nz = tgt[(tgt["bank_id"]==b) & (tgt["target_type"]=="net_zero")] if tgt is not None else None
    intt = tgt[(tgt["bank_id"]==b) & (tgt["target_type"]=="intensity_reduction")] if tgt is not None else None
    nz_year = int(nz["target_year"].iloc[0]) if nz is not None and not nz.empty else None
    fw = nz["target_framework"].iloc[0] if nz is not None and not nz.empty else "NZBA"
    cp = scen[scen["bank_id"]==b]["carbon_price_assumption_eur_per_tco2e"] if scen is not None else None
    cp_lo = round(float(cp.min()),1) if cp is not None and len(cp) else None
    cp_hi = round(float(cp.max()),1) if cp is not None and len(cp) else None
    resourcing = round((fin_val(b,YEAR,"climate_capex_meur") or 0)+(fin_val(b,YEAR,"climate_opex_meur") or 0),2)
    tp.append(dict(
        tp_id=f"TP-{b}-{YEAR}", bank_id=b, reporting_year=YEAR, has_transition_plan=True,
        net_zero_target_year=nz_year, aligned_framework=fw,
        key_assumptions=(f"Carbon-price trajectory of EUR {cp_lo}-{cp_hi}/tCO2e across NGFS scenarios; "
                         f"orderly policy implementation toward {fw} pathways; gradual client transition "
                         f"in high-carbon sectors; stable renewable-technology cost decline."),
        dependencies=("Government policy and carbon-pricing stability; client/counterparty decarbonisation; "
                      "availability of credible transition finance demand; data quality on financed emissions."),
        resourcing_meur=resourcing,
        resourcing_description=("Funded from existing climate capex/opex budgets covering risk-management "
                                "capability, data and analytics, and green-product development."),
        prior_period_progress=("Interim milestones tracked against the intensity and absolute targets; "
                               "progress reported on a schedule basis pending verified reductions."),
        is_synthetic=True, data_source="synthetic_estimate"))
transition_plan = pd.DataFrame(tp)

# ---- 3) climate_financial_effects.csv  (IFRS S2 para 15-16) ----------------
LINE_ITEM = {"transition_market":"Loans and advances to customers - ECL allowance",
             "transition_policy":"Loans and advances to customers - ECL allowance",
             "transition_technology":"Loans and advances to customers - ECL allowance",
             "transition_reputational":"Net fee and commission income",
             "physical_acute":"Loans and advances - collateral value / ECL allowance",
             "physical_chronic":"Loans and advances - collateral value / ECL allowance"}
fe = []
if rr is not None:
    for b in banks:
        rrb = rr[(rr["bank_id"]==b)&(rr["reporting_year"]==YEAR)].sort_values("financial_impact_meur",ascending=False)
        for _, r in rrb.head(3).iterrows():
            imp = float(r["financial_impact_meur"]) if pd.notna(r["financial_impact_meur"]) else 0.0
            cat = str(r["risk_category"]); hor = str(r["time_horizon"])
            mat = (str(r.get("risk_rating","")).lower()=="high" and "short" in hor.lower())
            fe.append(dict(effect_id=f"FE-{r['risk_id']}", bank_id=b, reporting_year=YEAR,
                           driver_type="risk", linked_id=r["risk_id"],
                           affected_statement="balance_sheet/income_statement",
                           line_item=LINE_ITEM.get(cat,"Loans and advances to customers - ECL allowance"),
                           effect_timing="current", horizon=hor,
                           quantitative_effect_meur=round(imp*0.10,2),
                           qualitative_description=f"Current-period effect attributable to: {r['risk_name']}.",
                           material_adjustment_next_period_flag=bool(mat),
                           basis="modelled_estimate", is_synthetic=True))
            fe.append(dict(effect_id=f"FE-{r['risk_id']}-ANT", bank_id=b, reporting_year=YEAR,
                           driver_type="risk", linked_id=r["risk_id"],
                           affected_statement="balance_sheet/income_statement",
                           line_item=LINE_ITEM.get(cat,"Loans and advances to customers - ECL allowance"),
                           effect_timing="anticipated", horizon=hor,
                           quantitative_effect_meur=round(imp,2),
                           qualitative_description=f"Anticipated {hor} effect attributable to: {r['risk_name']}.",
                           material_adjustment_next_period_flag=bool(mat),
                           basis="modelled_estimate", is_synthetic=True))
    if opp is not None:
        for b in banks:
            ob = opp[(opp["bank_id"]==b)&(opp["reporting_year"]==YEAR)].sort_values("estimated_revenue_impact_meur",ascending=False)
            for _, o in ob.head(1).iterrows():
                rev=float(o["estimated_revenue_impact_meur"]) if pd.notna(o["estimated_revenue_impact_meur"]) else 0.0
                fe.append(dict(effect_id=f"FE-{o['opportunity_id']}", bank_id=b, reporting_year=YEAR,
                               driver_type="opportunity", linked_id=o["opportunity_id"],
                               affected_statement="income_statement",
                               line_item="Net interest income - green and sustainability-linked lending",
                               effect_timing="anticipated", horizon=str(o.get("time_horizon","")),
                               quantitative_effect_meur=round(rev,2),
                               qualitative_description=f"Anticipated revenue opportunity: {o['description']}",
                               material_adjustment_next_period_flag=False,
                               basis="estimate", is_synthetic=True))
climate_financial_effects = pd.DataFrame(fe)

# ---- 4) resilience_assessment.csv  (IFRS S2 para 22(a)) --------------------
RES = {"orderly":"Modelled losses are within current capital buffers; gradual portfolio reweighting feasible.",
       "disorderly":"Near-term resilience manageable; medium-term capital consumption from stranded assets may require additional buffers.",
       "hot_house":"Physical-risk losses dominate long term; collateral revaluation and exposure limits are the main levers."}
ra=[]
for b in banks:
    capex=fin_val(b,YEAR,"climate_capex_meur"); opex=fin_val(b,YEAR,"climate_opex_meur")
    for st in ["orderly","disorderly","hot_house"]:
        ra.append(dict(resilience_id=f"RES-{b}-{st}", bank_id=b, reporting_year=YEAR, scenario_type=st,
            capacity_to_adjust=RES[st],
            asset_redeployment_capacity=("Lending portfolio can be repriced and reweighted at origination/renewal; "
                                         "limited owned physical assets to redeploy."),
            financial_resource_flexibility=f"Climate capex EUR {capex}m and opex EUR {opex}m; buffers per ICAAP.",
            significant_uncertainties=("Policy/carbon-price path, counterparty transition speed, physical-hazard "
                                       "frequency, and financed-emissions data quality."),
            assessment_horizon="short_medium_long", is_synthetic=True, data_source="synthetic_estimate"))
resilience_assessment = pd.DataFrame(ra)

# ---- 5) ghg_methodology.csv + scope12_consolidation.csv (29(a)(iii),(iv)) --
METH = {
 "scope1":("GHG Protocol Corporate Standard; activity x emission factor","Facility fuel use, fleet fuel/distance",
           "Operational control boundary; IPCC AR6 GWPs","Direct measurement where metered, else activity-based"),
 "scope2":("GHG Protocol Corporate Standard; location- and market-based","Metered electricity, supplier/grid factors, RECs",
           "Location and market methods both computed","Dual reporting per GHG Protocol Scope 2 Guidance"),
 "scope3":("GHG Protocol Scope 3 Standard; spend-/average-data","Operating spend, headcount, average emission factors",
           "Estimation uncertainty higher; data-quality scored","Material categories prioritised"),
 "financed":("PCAF Global GHG Accounting Standard","Outstanding/EVIC attribution; counterparty emissions; sovereign PPP-GDP",
             "Attribution factor x counterparty emissions","Financed emissions are the dominant Scope 3 category"),
}
gm=[]
for b in banks:
    for sc,(approach,inputs,assumptions,reason) in METH.items():
        gm.append(dict(method_id=f"GM-{b}-{sc}", bank_id=b, reporting_year=YEAR, scope=sc,
            measurement_approach=approach, key_inputs=inputs, key_assumptions=assumptions,
            reason_for_approach=reason, changes_in_period="None", consolidation_basis="operational_control",
            gwp_basis="IPCC_AR6", standard_reference=("GHG Protocol" if sc!="financed" else "PCAF"),
            is_synthetic=True))
ghg_methodology=pd.DataFrame(gm)

cons=[]
for b in banks:
    for y in years:
        for sc in ["scope1","scope2"]:
            cons.append(dict(cons_id=f"CON-{b}-{y}-{sc}", bank_id=b, reporting_year=y, scope=sc,
                consolidated_group_share_pct=100.0, other_investees_share_pct=0.0,
                consolidation_basis="operational_control",
                note="Operational emissions relate to the consolidated accounting group; no material associate/JV operational emissions.",
                is_synthetic=True))
scope12_consolidation=pd.DataFrame(cons)

# ---- write -----------------------------------------------------------------
NEW = {"scope3_categories":(scope3_categories,"S2 29(a)(i)(3) & 29(a)(vi) - Scope 3 categories"),
       "transition_plan":(transition_plan,"S2 14(a)(iv),(b),(c) - transition plan assumptions/resourcing/progress"),
       "climate_financial_effects":(climate_financial_effects,"S2 15-16 - financial effects by line item"),
       "resilience_assessment":(resilience_assessment,"S2 22(a) - resilience capacity & uncertainties"),
       "ghg_methodology":(ghg_methodology,"S2 29(a)(iii) - GHG measurement methodology"),
       "scope12_consolidation":(scope12_consolidation,"S2 29(a)(iv) - Scope 1&2 consolidation split")}
manifest=[]
for name,(df,why) in NEW.items():
    df.to_csv(gd/f"{name}.csv", index=False)
    manifest.append({"file":f"{name}.csv","rows":len(df),"banks":df["bank_id"].nunique(),"addresses":why})
Path("missing_data_manifest.json").write_text(json.dumps(manifest,indent=2))

print("Created new tables in", gd.resolve())
for m in manifest:
    print(f"  {m['file']:30s} {m['rows']:>4d} rows  ({m['banks']} banks)  -> {m['addresses']}")
print("\nAll rows flagged is_synthetic=True. Existing CSVs untouched. Manifest: missing_data_manifest.json")


Created new tables in C:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data
  scope3_categories.csv           225 rows  (5 banks)  -> S2 29(a)(i)(3) & 29(a)(vi) - Scope 3 categories
  transition_plan.csv               5 rows  (5 banks)  -> S2 14(a)(iv),(b),(c) - transition plan assumptions/resourcing/progress
  climate_financial_effects.csv    35 rows  (5 banks)  -> S2 15-16 - financial effects by line item
  resilience_assessment.csv        15 rows  (5 banks)  -> S2 22(a) - resilience capacity & uncertainties
  ghg_methodology.csv              20 rows  (5 banks)  -> S2 29(a)(iii) - GHG measurement methodology
  scope12_consolidation.csv        30 rows  (5 banks)  -> S2 29(a)(iv) - Scope 1&2 consolidation split

All rows flagged is_synthetic=True. Existing CSVs untouched. Manifest: missing_data_manifest.json
